In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgboost
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,mean_squared_error,f1_score,recall_score,precision_score


In [7]:
df = pd.read_csv('crypto_top1000_dataset.csv')

In [8]:
df.head()

,id,symbol,name,market_cap_rank,current_price,market_cap,fully_diluted_valuation,total_volume,high_24h,low_24h,...,price_change_percentage_24h,price_change_percentage_1h,price_change_percentage_7d,price_change_percentage_30d,price_change_percentage_1y,market_cap_change_24h,market_cap_change_percentage_24h,last_updated,image,supply_utilization
0,bitcoin,BTC,Bitcoin,1,92551.00,1849586398531,1.849586e+12,8.165691e+10,93929.00,90837.00,...,0.66511,-0.473153,5.391199,-13.158877,-2.776377,1.411594e+10,0.76906,2025-12-03T18:57:01.835Z,https://coin-images.coingecko.com/coins/images...,95.034210
1,ethereum,ETH,Ethereum,2,3114.18,377137085161,3.771371e+11,2.534897e+10,3139.77,2976.94,...,3.29234,-0.530166,5.687871,-13.815571,-12.423514,1.290545e+10,3.54320,2025-12-03T18:57:03.636Z,https://coin-images.coingecko.com/coins/images...,0.000000
2,tether,USDT,Tether,3,1.00,184744825283,1.902134e+11,1.017624e+11,1.00,1.00,...,0.00099,-0.004542,0.043571,0.030966,0.013444,1.066272e+08,0.05775,2025-12-03T18:57:02.158Z,https://coin-images.coingecko.com/coins/images...,0.000000
3,ripple,XRP,XRP,4,2.17,131451182003,2.178500e+11,4.038903e+09,2.21,2.14,...,0.01637,-0.497115,-0.706636,-8.403034,-13.724117,3.666673e+08,0.27972,2025-12-03T18:57:00.544Z,https://coin-images.coingecko.com/coins/images...,60.331635
4,binancecoin,BNB,BNB,5,902.22,124271801383,1.242718e+11,2.001782e+09,908.27,873.60,...,2.40889,-0.549929,3.466731,-9.351678,39.560349,2.885939e+09,2.37749,2025-12-03T18:57:01.669Z,https://coin-images.coingecko.com/coins/images...,68.868065


In [9]:
df.describe()

,market_cap_rank,current_price,market_cap,fully_diluted_valuation,total_volume,high_24h,low_24h,circulating_supply,total_supply,max_supply,...,atl_change_percentage,price_change_24h,price_change_percentage_24h,price_change_percentage_1h,price_change_percentage_7d,price_change_percentage_30d,price_change_percentage_1y,market_cap_change_24h,market_cap_change_percentage_24h,supply_utilization
count,1000.000000,1.000000e+03,1.000000e+03,9.980000e+02,1.000000e+03,9.870000e+02,9.870000e+02,1.000000e+03,9.980000e+02,5.650000e+02,...,1.000000e+03,987.000000,987.000000,980.000000,995.000000,978.000000,677.000000,9.860000e+02,986.000000,1000.000000
mean,500.514000,3.020196e+03,3.382915e+09,5.688250e+09,2.693727e+08,3.085377e+03,2.975851e+03,1.856249e+14,4.335387e+14,7.658999e+14,...,1.447517e+32,40.806696,0.898115,-0.086534,273.188205,242.640769,89.565864,4.067557e+07,32.768743,34.619589
std,288.842694,1.577453e+04,6.032522e+10,8.648060e+10,4.221592e+09,1.607505e+04,1.551608e+04,5.513662e+15,1.329493e+16,1.766923e+16,...,4.577452e+33,229.194971,6.384397,1.224800,8582.654705,7677.341273,2072.411310,6.199682e+08,995.592805,38.952744
min,1.000000,7.392680e-10,2.579177e+07,2.579177e+07,0.000000e+00,7.552700e-10,6.990220e-10,2.890172e+02,0.000000e+00,8.888000e+03,...,0.000000e+00,-3.942972,-37.457900,-8.695587,-64.142203,-95.802529,-99.981755,-3.840779e+08,-36.528110,0.000000
25%,250.750000,5.445100e-02,4.126963e+07,5.596051e+07,3.811775e+05,5.612500e-02,5.209250e-02,4.159729e+07,4.802165e+07,4.000000e+08,...,1.850787e+01,-0.000496,-0.733705,-0.441268,-3.081141,-17.980009,-79.229659,-5.840339e+05,-0.817982,0.000000
50%,500.500000,4.194065e-01,7.931666e+07,1.302084e+08,4.649918e+06,4.204880e-01,3.944960e-01,3.349151e+08,9.541714e+08,1.000000e+09,...,9.391917e+01,0.000191,0.210030,-0.122374,0.064689,-10.840300,-55.124787,3.379245e+05,0.387745,18.840967
75%,750.250000,1.965000e+00,2.440914e+08,3.908282e+08,1.953813e+07,1.840000e+00,1.770000e+00,1.709020e+09,2.662840e+09,1.000000e+10,...,5.556947e+02,0.014306,2.432890,0.058779,3.558208,0.028362,-1.778249,3.088509e+06,2.576065,73.009876
max,1002.000000,9.816900e+04,1.849586e+12,1.950679e+12,1.017624e+11,9.991500e+04,9.501400e+04,1.743388e+17,4.200000e+17,4.200000e+17,...,1.447517e+35,2480.660000,111.765830,21.264514,270728.629225,240068.374988,50754.529716,1.411594e+10,31262.617420,100.000000


In [12]:
df['volume_to_marketcap'] = df['total_volume'] / df['market_cap']

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                1000 non-null   str    
 1   symbol                            1000 non-null   str    
 2   name                              1000 non-null   str    
 3   market_cap_rank                   1000 non-null   int64  
 4   current_price                     1000 non-null   float64
 5   market_cap                        1000 non-null   int64  
 6   fully_diluted_valuation           998 non-null    float64
 7   total_volume                      1000 non-null   float64
 8   high_24h                          987 non-null    float64
 9   low_24h                           987 non-null    float64
 10  circulating_supply                1000 non-null   float64
 11  total_supply                      998 non-null    float64
 12  max_supply        

In [14]:
df.columns.tolist()

['id',
 'symbol',
 'name',
 'market_cap_rank',
 'current_price',
 'market_cap',
 'fully_diluted_valuation',
 'total_volume',
 'high_24h',
 'low_24h',
 'circulating_supply',
 'total_supply',
 'max_supply',
 'ath',
 'ath_change_percentage',
 'ath_date',
 'atl',
 'atl_change_percentage',
 'atl_date',
 'price_change_24h',
 'price_change_percentage_24h',
 'price_change_percentage_1h',
 'price_change_percentage_7d',
 'price_change_percentage_30d',
 'price_change_percentage_1y',
 'market_cap_change_24h',
 'market_cap_change_percentage_24h',
 'last_updated',
 'image',
 'supply_utilization',
 'volume_to_marketcap']

In [16]:
df.head()

,id,symbol,name,market_cap_rank,current_price,market_cap,fully_diluted_valuation,total_volume,high_24h,low_24h,...,price_change_percentage_1h,price_change_percentage_7d,price_change_percentage_30d,price_change_percentage_1y,market_cap_change_24h,market_cap_change_percentage_24h,last_updated,image,supply_utilization,volume_to_marketcap
0,bitcoin,BTC,Bitcoin,1,92551.00,1849586398531,1.849586e+12,8.165691e+10,93929.00,90837.00,...,-0.473153,5.391199,-13.158877,-2.776377,1.411594e+10,0.76906,2025-12-03T18:57:01.835Z,https://coin-images.coingecko.com/coins/images...,95.034210,0.044149
1,ethereum,ETH,Ethereum,2,3114.18,377137085161,3.771371e+11,2.534897e+10,3139.77,2976.94,...,-0.530166,5.687871,-13.815571,-12.423514,1.290545e+10,3.54320,2025-12-03T18:57:03.636Z,https://coin-images.coingecko.com/coins/images...,0.000000,0.067214
2,tether,USDT,Tether,3,1.00,184744825283,1.902134e+11,1.017624e+11,1.00,1.00,...,-0.004542,0.043571,0.030966,0.013444,1.066272e+08,0.05775,2025-12-03T18:57:02.158Z,https://coin-images.coingecko.com/coins/images...,0.000000,0.550827
3,ripple,XRP,XRP,4,2.17,131451182003,2.178500e+11,4.038903e+09,2.21,2.14,...,-0.497115,-0.706636,-8.403034,-13.724117,3.666673e+08,0.27972,2025-12-03T18:57:00.544Z,https://coin-images.coingecko.com/coins/images...,60.331635,0.030725
4,binancecoin,BNB,BNB,5,902.22,124271801383,1.242718e+11,2.001782e+09,908.27,873.60,...,-0.549929,3.466731,-9.351678,39.560349,2.885939e+09,2.37749,2025-12-03T18:57:01.669Z,https://coin-images.coingecko.com/coins/images...,68.868065,0.016108


In [17]:
y = np.where(df['price_change_percentage_1y'] > 0,1,0)

In [30]:
X = df.drop(columns=[
    'price_change_percentage_1y', 'name', 'symbol', 'id', 'image', 
    'ath_date', 'atl_date', 'last_updated',
    'current_price', 'high_24h', 'low_24h', 'price_change_24h', 'atl',
    'fully_diluted_valuation', 'market_cap_change_24h', 'market_cap_change_percentage_24h',
    'total_supply', 'max_supply','ath'
])

In [24]:
print(y)

[0 0 1 0 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 1 1 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0
 0 0 1 0 0 0 1 1 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1
 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0
 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0
 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1
 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0
 0 1 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0
 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 1 0 0 0 0 

In [31]:
print(X)

     market_cap_rank     market_cap  total_volume  circulating_supply  \
0                  1  1849586398531  8.165691e+10        1.995718e+07   
1                  2   377137085161  2.534897e+10        1.206954e+08   
2                  3   184744825283  1.017624e+11        1.846881e+11   
3                  4   131451182003  4.038903e+09        6.033164e+10   
4                  5   124271801383  2.001782e+09        1.377361e+08   
..               ...            ...           ...                 ...   
995              999       25835208  4.341250e+03        8.207440e+07   
996             1000       25832950  5.328143e+06        2.508233e+06   
997             1001       25816526  7.819600e+04        2.211393e+07   
998             1002       25815570  1.230519e+07        2.582835e+07   
999              997       25791773  6.185000e+04        3.491824e+06   

     ath_change_percentage  atl_change_percentage  \
0                -26.16251           1.371889e+05   
1                

In [32]:
print(list(X.columns))

['market_cap_rank', 'market_cap', 'total_volume', 'circulating_supply', 'ath_change_percentage', 'atl_change_percentage', 'price_change_percentage_24h', 'price_change_percentage_1h', 'price_change_percentage_7d', 'price_change_percentage_30d', 'supply_utilization', 'volume_to_marketcap']


In [34]:
#TRAIN/TEST/SPLIT
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [35]:
print(X_train.shape)

(800, 12)


In [36]:
print(X_test.shape)

(200, 12)


In [44]:
pipeline = Pipeline([
    ('scaler',StandardScaler()),
     ('imputer', SimpleImputer(strategy='median')),
    ('classifer',LogisticRegression(random_state=42))
])

In [45]:
cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

In [46]:
cv_scores = cross_val_score(pipeline, X_train, y_train,cv=cv, scoring='accuracy')

In [47]:
print(f"CV Mean Accuracy: {cv_scores.mean():.2f}")

CV Mean Accuracy: 0.86
